In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import time
import os
import sys
import copy
import math
import matplotlib.pyplot as plt
from resnet import *
from mobilenet import *
from densenet import *
from torx.module.layer import *
from VGG import *

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.25.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [19]:
# config_file_path = os.path.join(os.path.dirname(__file__), f'../config/config.json')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet18 = ResNet18(num_classes=10)
resnet18.load_state_dict(torch.load('trained_models/resnet18_cifar10.pth'))
resnet18.to(device)
vgg16 = VGG('VGG16', num_classes=10)
vgg16.load_state_dict(torch.load('trained_models/vgg16_cifar10.pth'))
vgg16.to(device)
densenet=densenet121(num_classes=10) 
densenet.load_state_dict(torch.load('trained_models/densenet121_cifar10.pth'))
densenet.to(device)

DenseNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (features): Sequential(
    (dense_block_layer_0): Sequential(
      (bottle_neck_layer_0): Bottleneck(
        (bottle_neck): Sequential(
          (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): ReLU(inplace=True)
          (2): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (4): ReLU(inplace=True)
          (5): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
      )
      (bottle_neck_layer_1): Bottleneck(
        (bottle_neck): Sequential(
          (0): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (1): ReLU(inplace=True)
          (2): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (3): BatchNorm2d(12

In [3]:
params = []
print(len(params))
i=0
# print(resnet18.named_parameters())
for name, module in resnet18.named_modules():
    parent_name, child_name = name.rsplit('.', 1) if '.' in name else (None, name)
    if isinstance(module, nn.Conv2d) or isinstance(module,nn.Linear):
        # print(f'{name} is a Conv2d layer')
        # print/(name/)
        # for name,param in module.named_parameters():
        #     if name == 'weight':
        #         print(param.shape)
        #         params.append(param)
        
        params.append(module.weight)
        # break
        #print the shape of the weights
        # print(module.weight.shape)
        # i=i+1
        # print(i)
# print(len(params))
# print(params[0])

0


In [20]:
import torch
import torchvision
import torchvision.transforms as transforms

# Define a transform to normalize the data
mean = (0.4914, 0.4822, 0.4465)
std= (0.2470, 0.2435, 0.2616)
kwargs = {'num_workers': 2, 'pin_memory': True}

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

# Load the CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10( root='./data', train=False, download=True, transform=transform)

testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=False, **kwargs)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


Files already downloaded and verified


In [21]:
final_filterwise_trace =  torch.load('cifar-10/final_filterwise_trace_densenet.pt')
print(len(final_filterwise_trace))

121


In [22]:
import math
##Reshaping the each elemnt to crossbar shape
N=128 #crossbar size
P= 16 #precision of weight
B= 2 #resolution of reRAM cell
#No of output filters in crossbar placed across the rows 
Nofxb = N//(P//B)
#No of in channels in crossbar placed across the columns(depends on the kernel size)
# first get the kernel size for each layer in conv layers
kernel_size = []
Nic = [] #no of in channels
Noc= [] #no of out channels
conv_fc =[]
for name, module in densenet.named_modules():
    parent_name, child_name = name.rsplit('.', 1) if '.' in name else (None, name)
    if isinstance(module, nn.Conv2d):
        # print(module)
        kernel_size.append(module.kernel_size[0]*module.kernel_size[1])
        Nic.append(module.in_channels)
        Noc.append(module.out_channels)
        conv_fc.append(0)
    if isinstance(module, nn.Linear):
        kernel_size.append(1)
        Nic.append(module.in_features)
        Noc.append(module.out_features)
        conv_fc.append(1)
# print(kernel_size)
# for i in range(len(ftrace)):
#Reuirenumber of crossbars for each layer
Nofrowsxb = []
Nofcolxb = []
Nifxb = []
for i in range(len(final_filterwise_trace)):
    Nofcolxb.append(math.ceil(Noc[i]/Nofxb))
    Nofrowsxb.append(math.ceil(Nic[i]*kernel_size[i]/N))
    Nifxb.append(N/kernel_size[i])
print(Nofrowsxb)
print(Nofcolxb)
print(Nifxb)

[1, 1, 9, 1, 9, 1, 9, 2, 9, 2, 9, 2, 9, 2, 1, 9, 2, 9, 2, 9, 2, 9, 2, 9, 3, 9, 3, 9, 3, 9, 3, 9, 4, 9, 4, 9, 4, 9, 4, 2, 9, 3, 9, 3, 9, 3, 9, 3, 9, 4, 9, 4, 9, 4, 9, 4, 9, 5, 9, 5, 9, 5, 9, 5, 9, 6, 9, 6, 9, 6, 9, 6, 9, 7, 9, 7, 9, 7, 9, 7, 9, 8, 9, 8, 9, 8, 9, 8, 4, 9, 5, 9, 5, 9, 5, 9, 5, 9, 6, 9, 6, 9, 6, 9, 6, 9, 7, 9, 7, 9, 7, 9, 7, 9, 8, 9, 8, 9, 8, 9, 8]
[4, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 16, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 32, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 8, 2, 1]
[14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 128.0, 14.222222222222221, 128.0, 14.222222222222221, 128.0, 14.222222222222221,

In [23]:
print(kernel_size)
print(Nic)
import torch
import torch.nn as nn

# # Define the number of input and output features
# D_in = 10  # Number of input features
# D_out = 5  # Number of output features

# # Define the linear layer
# linear_layer = nn.Linear(D_in, D_out)

# # Get the shape of the weights
# weights_shape = linear_layer.weight.shape

# print("Shape of weights:", weights_shape)


[9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1, 9, 1]
[3, 64, 128, 96, 128, 128, 128, 160, 128, 192, 128, 224, 128, 256, 128, 128, 160, 128, 192, 128, 224, 128, 256, 128, 288, 128, 320, 128, 352, 128, 384, 128, 416, 128, 448, 128, 480, 128, 512, 256, 128, 288, 128, 320, 128, 352, 128, 384, 128, 416, 128, 448, 128, 480, 128, 512, 128, 544, 128, 576, 128, 608, 128, 640, 128, 672, 128, 704, 128, 736, 128, 768, 128, 800, 128, 832, 128, 864, 128, 896, 128, 928, 128, 960, 128, 992, 128, 1024, 512, 128, 544, 128, 576, 128, 608, 128, 640, 128, 672, 128, 704, 128, 736, 128, 768, 128, 800, 128, 832, 128, 864, 128, 896, 128, 928, 128, 960, 128, 992, 128, 1024]


In [24]:
filter_groups_row = []
index_of_xb_to_start_from = {}
no_of_filter_in_xb_col = {}
# no_of_intial_row_fixed = {}
indication_to_pick_prvious_filter = {}
for i in range(len(final_filterwise_trace)):
    print(i)
    # for i in range(len(ftrace)):
    #split the list into sublist every Nic elements in each ftrace
    # no_of_intial_row_fixed[f'layer_{i}'] = []
    no_of_filter_in_xb_col[f'layer_{i}'] = []
    indication_to_pick_prvious_filter[f'layer_{i}'] = []
    index_of_xb_to_start_from[f'layer_{i}'] = []
    index = index_of_xb_to_start_from[f'layer_{i}']
    NIRF = []
    NFXBC = no_of_filter_in_xb_col[f'layer_{i}']
    ITPF = indication_to_pick_prvious_filter[f'layer_{i}']
    # print(ftrace[i])
    # split_ftrace[i]=
    if Nic[i]*kernel_size[i] > N:
        no_of_xbs = math.ceil(Nic[i]*kernel_size[i]/N)
        print('No of crossbars required is',no_of_xbs, 'for layer',i )
        NIRF.append(0)
        index.append(0)
        # ITPF.append(0)
        
        for j in range(no_of_xbs):
            no_of_rows_left_to_map = N - NIRF[j]
            print(no_of_rows_left_to_map)
            print(kernel_size[i])
            if no_of_rows_left_to_map%kernel_size[i] != 0:
                NIRF.append(kernel_size[i]-no_of_rows_left_to_map % kernel_size[i])
                index.append(kernel_size[i]-NIRF[j+1])
            else:
                NIRF.append(0)
                index.append(0)
            # NIRF.append(kernel_size[i]-no_of_rows_left_to_map % kernel_size[i])
            if NIRF[j]>0:
                NFXBC.append(math.ceil((no_of_rows_left_to_map)/kernel_size[i])+1)
            else:
                NFXBC.append(math.ceil((no_of_rows_left_to_map)/kernel_size[i]))
            if no_of_rows_left_to_map%kernel_size[i] == 0 or kernel_size[i] %no_of_rows_left_to_map == 0:
                ITPF.append(0)
            else:
                ITPF.append(1)
    else:
        print('No of crossbars required is 1 for layer',i )
        NFXBC.append(math.ceil((N)/kernel_size[i]))
        ITPF.append(0)
        index.append(0)
        #do padding
            # no_cells_included_in_xb.append(((N-kernel_size[i]+no_cells_included_in_xb[j])%kernel_size[i]))


0
No of crossbars required is 1 for layer 0
1
No of crossbars required is 1 for layer 1
2
No of crossbars required is 9 for layer 2
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
3
No of crossbars required is 1 for layer 3
4
No of crossbars required is 9 for layer 4
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
5
No of crossbars required is 1 for layer 5
6
No of crossbars required is 9 for layer 6
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
7
No of crossbars required is 2 for layer 7
128
1
128
1
8
No of crossbars required is 9 for layer 8
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
9
No of crossbars required is 2 for layer 9
128
1
128
1
10
No of crossbars required is 9 for layer 10
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
11
No of crossbars required is 2 for layer 11
128
1
128
1
12
No of crossbars required is 9 for layer 12
128
9
121
9
123
9
125
9
127
9
120
9
122
9
124
9
126
9
13
No of crossbars required is 2 for layer 13
128
1
128
1
14
N

In [25]:
print(index_of_xb_to_start_from)
print(no_of_filter_in_xb_col)   
print(indication_to_pick_prvious_filter)
# print(split_ftrace[0])

{'layer_0': [0], 'layer_1': [0], 'layer_2': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_3': [0], 'layer_4': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_5': [0], 'layer_6': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_7': [0, 0, 0], 'layer_8': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_9': [0, 0, 0], 'layer_10': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_11': [0, 0, 0], 'layer_12': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_13': [0, 0, 0], 'layer_14': [0], 'layer_15': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_16': [0, 0, 0], 'layer_17': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_18': [0, 0, 0], 'layer_19': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_20': [0, 0, 0], 'layer_21': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_22': [0, 0, 0], 'layer_23': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_24': [0, 0, 0, 0], 'layer_25': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_26': [0, 0, 0, 0], 'layer_27': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_28': [0, 0, 0, 0], 'layer_29': [0, 2, 4, 6, 8, 1, 3, 5, 7, 0], 'layer_30': [0, 0, 0, 0], 'layer_31': [0, 2, 4

In [26]:
ftrace_ic = []
padded_layer = {}
# print(len(ftrace[20]))
for i in range(len(final_filterwise_trace)):
    #split the list into sublist every Nic elements in each ftrace
    # print(ftrace[i])
    if conv_fc[i] == 0:
        ftrace_ic.append( [final_filterwise_trace[i][j * Nic[i]:(j + 1) * Nic[i]] for j in range(0,  len(final_filterwise_trace[i])//Nic[i])])
    else:
        # print(final_filterwise_trace[i])
        ftrace_ic.append([final_filterwise_trace[i]])
# print(len(ftrace_ic))
for i in range(len(ftrace_ic)):
    padded_layer[f'layer_{i}'] = []

    # print(i)
    NIRF= no_of_filter_in_xb_col[f'layer_{i}']
    if i ==20: print(NIRF)
    ITPF = indication_to_pick_prvious_filter[f'layer_{i}']
    if i == 20: print(len(ftrace_ic[i]))
    ##This is for each layer you do the grouping of the filters as per the groiuping from no_of_filter_in_xb_col
    padded_value = 0
    for j in range(len(ftrace_ic[i])):
       
        new_list = []
        # print(len(ftrace_ic[i][j]))
        #Repeating the the process for each out channel filters in the layer
        
        for k in range(len(NIRF)):
            
            if k==0:
                low_index = 0
            else:                    
                low_index = sum(NIRF[:k]) - sum(ITPF[:k])
            
            if i == 20: print(low_index, len(ftrace_ic[i][j][low_index:]),NIRF[k])
            
            if len(ftrace_ic[i][j][low_index:]) >= NIRF[k]:
                new_list.append(ftrace_ic[i][j][low_index:NIRF[k]+low_index])
            else:
                #fill the remaining with zeros
                # print(NIRF[k]-len(ftrace_ic[i][j]))
                padded_value = NIRF[k]-len(ftrace_ic[i][j][low_index:])
                new_list.append(ftrace_ic[i][j][low_index:]+[0]*(NIRF[k]-len(ftrace_ic[i][j][low_index:])))
        if j==len(ftrace_ic[i])-1: padded_layer[f'layer_{i}'].append(padded_value)
    # if i==

    #     print(new_list)
        # print(len(new_list))        
        ftrace_ic[i][j] = new_list
    # if i==1:
    #     break

ranked_tensor_list = []

    #     else:
# print(ftrace_ic[1][0])
# for j in range(len(ftrace_ic[0][0])):
#     print(len(ftrace_ic[0][0][j]))
# print(len(ftrace_ic))  

[128, 128]
128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 128
0 224 128
128 96 12

In [27]:
# print((ftrace_ic[19][0]))

# tensor_new = torch.tensor(ftrace_ic[1][0]).T
# print(tensor_new.shape)
print(Nofxb)
print(padded_layer)

16
{'layer_0': [12], 'layer_1': [64], 'layer_2': [0], 'layer_3': [32], 'layer_4': [0], 'layer_5': [0], 'layer_6': [0], 'layer_7': [96], 'layer_8': [0], 'layer_9': [64], 'layer_10': [0], 'layer_11': [32], 'layer_12': [0], 'layer_13': [0], 'layer_14': [0], 'layer_15': [0], 'layer_16': [96], 'layer_17': [0], 'layer_18': [64], 'layer_19': [0], 'layer_20': [32], 'layer_21': [0], 'layer_22': [0], 'layer_23': [0], 'layer_24': [96], 'layer_25': [0], 'layer_26': [64], 'layer_27': [0], 'layer_28': [32], 'layer_29': [0], 'layer_30': [0], 'layer_31': [0], 'layer_32': [96], 'layer_33': [0], 'layer_34': [64], 'layer_35': [0], 'layer_36': [32], 'layer_37': [0], 'layer_38': [0], 'layer_39': [0], 'layer_40': [0], 'layer_41': [96], 'layer_42': [0], 'layer_43': [64], 'layer_44': [0], 'layer_45': [32], 'layer_46': [0], 'layer_47': [0], 'layer_48': [0], 'layer_49': [96], 'layer_50': [0], 'layer_51': [64], 'layer_52': [0], 'layer_53': [32], 'layer_54': [0], 'layer_55': [0], 'layer_56': [0], 'layer_57': [96]

In [28]:

def pad_and_slice_tensor(input_tensor, Nofxb):
    """
    Pad and slice a 2D tensor.

    Parameters:
    - tensor: A 2D tensor to be padded and sliced.
    - Nofxb: Desired number of slices along the second dimension.

    Returns:
    - A new tensor obtained by stacking the slices.
    """
    # Calculate the padding needed to make the second dimension divisible by Nofxb
    padding_needed = (Nofxb - (input_tensor.size(1) % Nofxb)) % Nofxb
    # print("chunksize",Nofxb)

    # Pad the tensor with zeros at the end of the second dimension (dim=1) if needed
    if padding_needed > 0:
        tensor_padded = F.pad(input_tensor, (0, padding_needed), "constant", 0)
    else:
        tensor_padded = input_tensor

    # Split the padded tensor into slices
    slices_xb = torch.chunk(tensor_padded, chunks=(input_tensor.size(1)+padding_needed)//Nofxb, dim=1)
    # print("slice shape",slices_xb[0].shape)

    # Stack the slices and transpose to get the new tensor
    new_tensor_xb = torch.stack(slices_xb, dim=0)

    return new_tensor_xb


tensor_layer = []
ftrace_xb = []
P=16
B=2
for i in range(len(Nofrowsxb)):
    transformed_list = [list(map(list, zip(*sublist))) for sublist in zip(*ftrace_ic[i])]
    # print(len(transformed_list))
    # new_list=[]
    # for xb in range(len(Nofrowsxb[i])): 
    #     new_list.append(ftrace_ic[i][:][xb])
    ftrace_xb.append(transformed_list)

for i in range(len(ftrace_xb)):
    tensor_xb = []
    for j in range(len(ftrace_xb[i])):
        tensor = torch.tensor(ftrace_xb[i][j])
        # print(tensor.shape)
        tensor_xb.append(tensor)
    tensor_layer.append(tensor_xb)
        # tensor_full_col.append()
    # tensor_xb_col.append(tensor_col)


tensor_padded_layer = []
for i in range(len(tensor_layer)):
    tensor_padded_xb=[]
    for j in range(len(tensor_layer[i])):
        tensor = tensor_layer[i][j]
        # print("input shape",tensor.shape)
        padded_reshaped_tensor=pad_and_slice_tensor(tensor, Nofxb)
        # print("padded",padded_reshaped_tensor.shape)
        tensor_padded_xb.append(padded_reshaped_tensor)

    tensor_padded_layer.append(tensor_padded_xb)
        # # print(tensor_layer[i][j].shape)
        # # if conv_fc[i] == 0:
        #     padding_needed = (Nofxb - (tensor_layer[i][j].size(1) % Nofxb)) % Nofxb

        #     # Pad the tensor with zeros at the end of the second dimension (dim=1)
        #     if padding_needed > 0:
        #         # Padding format for F.pad is (left, right, top, bottom) for 2D, but we need (0, padding_needed) for the last dimension
        #         tensor_padded = F.pad(tensor_layer[i][j], (0, padding_needed), "constant", 0)
        #     else:
        #         tensor_padded = tensor_layer[i][j]
        #     # print(Nofxb)
        #     # print(tensor_layer[i][j].shape)
        #     slices_xb = torch.chunk(tensor_padded, chunks= int(Nofxb), dim=1)
        #     # print(slices_xb[0].shape)
        #     # If you want to create a new tensor from these slices (e.g., stacking them)
        #     new_tensor_xb = torch.stack(slices_xb, dim=0).transpose(0, 2)
        #     if i==20: print(new_tensor_xb.shape)
        #     tensor_layer[i][j] = new_tensor_xb 
    # print((tensor_full_col[i]).shape)
# for i in range(len(tensor_col)):
#     tensor_row.append(torch.stack(tensor_col[i],dim=0))
# print(tensor_xb_col[0].shape)
# ranked_tensor_list = []
final_tensor_layer = []
for i in range(len(tensor_padded_layer)):
    final_tensor_list = []
    for j in range(len(tensor_padded_layer[i])):
        # print(j)
        ranked_tensor = torch.zeros_like(tensor_padded_layer[i][j], dtype=torch.float)
        tensor = tensor_padded_layer[i][j]
        # print(i,j)
        a_dim, b_dim, c_dim = tensor.shape
        for a in range(a_dim):
            # Extract the current slice
            slice_flat = tensor[a].flatten()
            # Sort the slice in descending order to get the indices
            sorted_values, sorted_indices = torch.sort(slice_flat, descending=True)
            # Create an empty tensor for ranks
            ranks = torch.empty_like(sorted_values, dtype=torch.float)
            # Assign ranks, handling ties
            current_rank = 1
            for m in range(len(sorted_values)):
                if m == 0 or sorted_values[m] != sorted_values[m - 1]:
                    ranks[sorted_indices[m]] = current_rank
                    current_rank += 1
                else:
                    ranks[sorted_indices[m]] = ranks[sorted_indices[m - 1]]
            # Reshape ranks to match the original slice shape
            ranked_tensor[a] = ranks.view(b_dim, c_dim)
        # print(ranked_tensor)
        # print( 'length',(len(tensor_padded_layer[i])-1))
        if j == (len(tensor_padded_layer[i])-1):
            # print("vbsdkjvbSIFDBVLIHsdbV")
            padded_value = padded_layer[f'layer_{i}']
            ranked_tensor[:,-padded_value[0]:,:] = 0
            # print("un_padded",ranked_tensor)
            # print(kernel_size[i])
        # print(ranked_tensor.shape)
        replicated_tensor = torch.repeat_interleave(ranked_tensor, repeats=kernel_size[i], dim=1)
        # print(replicated_tensor.shape)
        replicated_tensor_next = torch.repeat_interleave(replicated_tensor, repeats=P//B, dim=2)
        # print(replicated_tensor_next.shape)
        #DEFINE A normal TENSOR WITH None values the first two dimensions same as the original tensor and the last two dimensions as the crossbar size
        modified_tensor = torch.empty((replicated_tensor_next.shape[0],N,N))
        # print(index_of_xb_to_start_from[f'layer_{i}'])
        t = index_of_xb_to_start_from[f'layer_{i}'][0]
        # for n in range(replicated_tensor.shape[0]):
            # print(t,b)
            # new_tensor = torch.empty((j,i,N,replicated_tensor.shape[3]))
            # print(replicated_tensor[:,t-1:N+t,:].shape)
        modified_tensor[:,:,:] = replicated_tensor_next[:,t:N+t,:]
        # t = kernel_size[i]- replicated_tensor.shape[2]+N+t
        # if i == replicated_tensor.shape[0]-2:
        #     b=kernel_size[i]
        # else:
        #     b =  (N - (kernel_size[i] - t))%kernel_size[i]
        final_tensor_list.append(modified_tensor)
    # if i==1: break
    # print('padded',ranked_tensor)
    # break
    final_tensor_layer.append(final_tensor_list)
# print((split_ftrace[0]))
# print(len(tensor_list))

In [29]:
# for i in range(len(final_tensor_layer)):
#     print(len(final_tensor_layer[i]))
#     for j in range(len(final_tensor_layer[i])):
#         print(final_tensor_layer[i][j].shape)
print(final_tensor_layer[0][0].shape)
cascaded_tensor = []
for i in range(len(final_tensor_layer)):
    #stack the tensors along the first dimension
    stacked_tensor = torch.stack(final_tensor_layer[i],dim=0)
    cascaded_tensor.append(stacked_tensor)


torch.Size([4, 128, 128])


In [31]:
# print(cascaded_tensor[20].shape)
for i in range(len(cascaded_tensor)):
    print(cascaded_tensor[i].shape)
torch.save(cascaded_tensor, 'fault_map_densenet.pth')

torch.Size([1, 4, 128, 128])
torch.Size([1, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([1, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([1, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([1, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([2, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([3, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([3, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([3, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([3, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([4, 8, 128, 128])
torch.Size([9, 2, 128, 128])
torch.Size([4,

In [20]:
import torch

# Example tensor of size (x, y)
x, y = 15, 64 # Example dimensions
tensor_test = torch.randn(x, y)  # Create a random tensor for demonstration

# Parameters
z = 16  # Desired size of each slice along y
num_chunks = y // z  # Calculate the number of chunks

# Split the tensor into slices of size z along y dimension
slices = torch.chunk(tensor_test, num_chunks, dim=1)

# If you want to create a new tensor from these slices (e.g., stacking them)
new_tensor = torch.stack(slices, dim=0)  # This stacks along a new dimension

print("Original Tensor Shape:", tensor_test.shape)
print("New Tensor Shape:",  new_tensor.shape)


Original Tensor Shape: torch.Size([15, 64])
New Tensor Shape: torch.Size([4, 15, 16])


In [21]:
import torch

def rank_slices_4d_tensor(tensor):
    a_dim, b_dim, c_dim, d_dim = tensor.shape
    ranked_tensor = torch.zeros_like(tensor, dtype=torch.float)

    for a in range(a_dim):
        for b in range(b_dim):
            # Extract the current slice
            slice_flat = tensor[a, b].flatten()
            # Sort the slice in descending order to get the indices
            sorted_values, sorted_indices = torch.sort(slice_flat, descending=True)
            # Create an empty tensor for ranks
            ranks = torch.empty_like(sorted_values, dtype=torch.float)
            # Assign ranks, handling ties
            current_rank = 1
            for i in range(len(sorted_values)):
                if i == 0 or sorted_values[i] != sorted_values[i - 1]:
                    ranks[sorted_indices[i]] = current_rank
                    current_rank += 1
                else:
                    ranks[sorted_indices[i]] = ranks[sorted_indices[i - 1]]
            # Reshape ranks to match the original slice shape
            ranked_tensor[a, b] = ranks.view(c_dim, d_dim)

    return ranked_tensor

# Example tensor with random integers
A = torch.randint(0, 10, (5, 2, 4, 4), dtype=torch.float)

# Rank the slices in descending order, ensuring identical values within each slice get the same rank
ranked_slices_desc = rank_slices_4d_tensor(A)

print("Original Tensor:\n", A)
print("\nRanked Tensor:\n", ranked_slices_desc)


Original Tensor:
 tensor([[[[9., 6., 7., 3.],
          [1., 0., 9., 6.],
          [8., 8., 3., 8.],
          [0., 4., 7., 4.]],

         [[9., 8., 7., 7.],
          [7., 6., 8., 0.],
          [7., 5., 0., 3.],
          [5., 4., 0., 7.]]],


        [[[3., 2., 8., 2.],
          [3., 0., 8., 9.],
          [5., 5., 4., 1.],
          [8., 2., 9., 0.]],

         [[1., 0., 3., 0.],
          [5., 5., 2., 9.],
          [2., 9., 9., 9.],
          [6., 3., 7., 1.]]],


        [[[5., 8., 2., 1.],
          [1., 2., 3., 4.],
          [0., 8., 9., 0.],
          [1., 5., 3., 9.]],

         [[2., 1., 9., 0.],
          [2., 3., 6., 5.],
          [7., 3., 2., 8.],
          [0., 4., 0., 5.]]],


        [[[1., 7., 9., 6.],
          [9., 2., 9., 7.],
          [7., 8., 0., 8.],
          [0., 5., 0., 6.]],

         [[6., 2., 6., 3.],
          [9., 8., 9., 5.],
          [9., 2., 0., 8.],
          [7., 2., 0., 7.]]],


        [[[0., 3., 8., 4.],
          [0., 4., 2., 4.],
      

In [22]:
import torch

# Example tensor of shape (a, b, c, d)
a, b, c, d = 5,1, 15, 16  # Example dimensions
Y = 2  # Replication factor

# Create an example tensor with random values
tensor = torch.randint(0,5,(a, b, c, d))

# Replicate each row 'c' 'Y' times for each 'b', 'a'
replicated_tensor = torch.repeat_interleave(tensor, repeats=9, dim=2)
replicated_tensor = torch.repeat_interleave(replicated_tensor, repeats=8, dim=3)
print(replicated_tensor.shape)
#DEFINE A normal TENSOR WITH None values the first two dimensions same as the original tensor and the last two dimensions as the crossbar size
modified_tensor = torch.empty((replicated_tensor.shape[0],replicated_tensor.shape[1],N,N))
t,b = 0,kernel_size[0]-N % kernel_size[0]
for i in range(replicated_tensor.shape[0]):
    print(i)
    print(replicated_tensor.shape[0])
    print(t,b)
    # new_tensor = torch.empty((j,i,N,replicated_tensor.shape[3]))
    print(replicated_tensor[i,:,t:replicated_tensor.shape[2]+1,:].shape)
    if i == replicated_tensor.shape[0]-1:
        modified_tensor[i,:,:,:] = replicated_tensor[i,:,t:replicated_tensor.shape[2],:]
    else:
        modified_tensor[i,:,:,:] = replicated_tensor[i,:,t:replicated_tensor.shape[2]-b,:]
    t = kernel_size[0]- b
    b = kernel_size[0]-(N-b)%kernel_size[0]
    

# Show the shape of the original and replicated tensor for verification
print("Original tensor shape:", tensor[1,0,0:2,:])
print("Replicated tensor shape:", modified_tensor[1,0,0:8,0:9])


torch.Size([5, 1, 135, 128])
0
5
0 7
torch.Size([1, 135, 128])
1
5
2 5
torch.Size([1, 133, 128])
2
5
4 3
torch.Size([1, 131, 128])
3
5
6 1
torch.Size([1, 129, 128])
4
5
8 8
torch.Size([1, 127, 128])


RuntimeError: The expanded size of the tensor (128) must match the existing size (127) at non-singleton dimension 1.  Target sizes: [1, 128, 128].  Tensor sizes: [127, 128]

In [ ]:
import torch
import torch.nn.functional as F

# Step 1: Create an initial tensor
initial_tensor = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])

# Step 2: Pad the tensor
pad_dim1, pad_dim2 = 1, 1  # Padding dimensions
padded_tensor = F.pad(initial_tensor, (0, pad_dim2, 0, pad_dim1), "constant", 0)

# Simulate a ranking operation (for demonstration, let's increase each element by 10)
# In a real scenario, this would be the result of your ranking operation
ranked_tensor = padded_tensor + 10

# Print the ranked tensor before resetting padded areas
print("Ranked tensor with padding:\n", ranked_tensor)

# Step 4: Reset ranks in padded areas
# Set the last 'pad_dim1' rows to 0
if pad_dim1 > 0:
    ranked_tensor[:, :, -pad_dim1:, :] = 0

# Set the last 'pad_dim2' columns to 0
if pad_dim2 > 0:
    ranked_tensor[:, :, :, -pad_dim2:] = 0

# Print the ranked tensor after resetting padded areas
print("\nRanked tensor after resetting padded areas:\n", ranked_tensor)


In [ ]:
import torch

# Example tensor of shape (a, b, c, d)
a, b, c, d = 2,1, 2, 3  # Example dimensions
Y = 2  # Replication factor

# Create an example tensor with random values
tensor = torch.randint(0,5,(a, b, c, d))

# Replicate each row 'c' 'Y' times for each 'b', 'a'
replicated_tensor = torch.repeat_interleave(tensor, repeats=9, dim=2)
replicated_tensor = torch.repeat_interleave(replicated_tensor, repeats=8, dim=3)
print(tensor)
print(replicated_tensor.shape)
print(replicated_tensor[0,:,0:16,:].shape)
print(replicated_tensor[0,:,0:16,:])

In [ ]:
import math

# Function to insert values into a list


# Example usage
list_of_sublists = [[1, 2, 3, 4], [5, 6, 7, 8, 9]]
list_of_Y_values = [2, 3.5]

modified_list_of_sublists = [
    insert_values(sublist, Y)
    for sublist, Y in zip(list_of_sublists, list_of_Y_values)
]

print(modified_list_of_sublists)


In [ ]:
import torch

# Original list of sublists
list_of_sublists = [[1,2,3],[5,6,7]]

# Transpose the sublists to form a tensor
tensor = torch.tensor(list_of_sublists).T

# Display the tensor
tensor
print(tensor.shape)

In [ ]:
import torch

def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    processed_images = 0  # Counter for the number of images processed

    with torch.no_grad():  # No need to calculate gradients
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)  # Move data to the device
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            processed_images += labels.size(0)  # Update the processed images counter
            
            # Check if the processed images counter has reached a multiple of 100
            if processed_images % 100 == 0:
                print(f'Processed {processed_images} images')

    accuracy = 100 * correct / total
    return accuracy




In [ ]:
accuracy = calculate_accuracy(resnet18, testloader,device)
print(f'Accuracy of the model on the 10000 test images: {accuracy}%')

In [ ]:
from scipy.spatial import KDTree

# Example dimensions for a larger matrix (e.g., 100x100 for demonstration)
c, d = 10, 10
# Example fault percentage
z = 25  # Example percentage

# Generate a 2-D grid of points
x = np.arange(c)
y = np.arange(d)
xx, yy = np.meshgrid(x, y)
points = np.c_[xx.ravel(), yy.ravel()]

# Construct a KD-Tree with the points
tree = KDTree(points)

# Select a random center from the points
random_center_index = np.random.choice(len(points))
random_center = points[random_center_index]

# Calculate the number of points to select based on z%
total_points = c * d
points_to_select = int((z / 100) * total_points)

# Query the KD-Tree for the nearest 'points_to_select' points to the random center
_, indices = tree.query(random_center, k=points_to_select)

# Initialize an array to mark clustered (1) and unclustered (0) points
clustered_points = np.zeros(total_points, dtype=int)

# Mark the selected points as clustered
clustered_points[indices] = 1

# Reshape the array to match the grid's dimensions for visualization
clustered_points_matrix = clustered_points.reshape(c, d)

# Show a snippet of the matrix to verify
clustered_points_matrix[:10, :10]  # Displaying just the top-left 10x10 portion for brevity


In [ ]:
from sklearn.mixture import GaussianMixture

# Correctly redefining the setup for Gaussian Mixture Model clustering
c, d = 16, 16  # Grid dimensions for a manageable demonstration
z = 60 # Fault percentage

# Generate grid points again
x, y = np.arange(c), np.arange(d)
xx, yy = np.meshgrid(x, y)
points = np.c_[xx.ravel(), yy.ravel()]

# Select a random center from the points
np.random.seed(0)  # For reproducibility
random_center_index = np.random.choice(len(points))
random_center = points[random_center_index]

# Initialize the Gaussian Mixture Model with the corrected means_init parameter
gmm = GaussianMixture(n_components=1, means_init=[random_center]).fit(points)

# Calculate the probabilities for each point
probabilities = gmm.score_samples(points)

# Find the threshold to include the desired fault percentage of points
threshold = np.percentile(probabilities, 100 - z)

# Determine clustered points based on the threshold
is_clustered = np.where(probabilities >= threshold, 1, 0)

# Reshape the clustered points indicator to match the grid's dimensions
clustered_indicator_matrix = is_clustered.reshape(c, d)

clustered_indicator_matrix


In [16]:
import torch
import numpy as np
from sklearn.mixture import GaussianMixture

class OptimizedClass:
    def __init__(self, p_state_shape, fault_rate, device='cpu'):
        self.p_state = torch.randn(p_state_shape, device=device)  # Initialize with the given shape on the specified device
        self.fault_rate = fault_rate
        self.device = device
        self.G_shape = p_state_shape  # Assuming G_shape is meant to match p_state_shape
        self.p_SA00 = [0.25]  # Example probability distribution
        self.p_SA01 = [0.25]
        self.p_SA10 = [0.25]
        self.p_SA11 = [0.25]

    def dist_gen_cluster(self):
        final_fault_state = torch.zeros(self.G_shape, device=self.device)
        
        max_points = int(self.fault_rate * self.p_state.numel())
        
        # Adjusting dimensions for meshgrid generation
        N, C, H, W = self.p_state.shape
        flat_dim_x, flat_dim_y = N * H, C * W
        

        # Correctly redefining the setup for Gaussian Mixture Model clustering
        c, d = flat_dim_x, flat_dim_y  # Grid dimensions for a manageable demonstration
        z = self.fault_rate * 100 # Fault percentage

        # Generate grid points again
        x, y = np.arange(c), np.arange(d)
        xx, yy = np.meshgrid(x, y)
        points = np.c_[xx.ravel(), yy.ravel()]

        # Select a random center from the points
        # np.random.seed(0)  # For reproducibility
        random_center_index = np.random.choice(len(points))
        random_center = points[random_center_index]

        # Initialize the Gaussian Mixture Model with the corrected means_init parameter
                # Initialize the Gaussian Mixture Model
        gmm = GaussianMixture(n_components=1, random_state=0, means_init=random_center.reshape(1, -1))
        gmm.fit(points)
        probabilities = gmm.score_samples(points)
        threshold = np.percentile(probabilities, 100 - self.fault_rate * 100)
        is_clustered = probabilities >= threshold
        # gmm = GaussianMixture(n_components=1, means_init=[random_center]).fit(points)

        # # Calculate the probabilities for each point
        # probabilities = gmm.score_samples(points)

        # # Find the threshold to include the desired fault percentage of points
        # threshold = np.percentile(probabilities, 100 - z)

        # # Determine clustered points based on the threshold
        # is_clustered = np.where(probabilities >= threshold, 1, 0)

        # Reshape the clustered points indicator to match the grid's dimensions
        clustered_indicator_matrix = is_clustered.reshape(c, d)
        print(clustered_indicator_matrix)
        # Convert the clustered indicator matrix to a PyTorch tensor and reshape it back to the original shape
        clustered_indicator_tensor = torch.tensor(clustered_indicator_matrix, device=self.device)
        intermediate_tensor = clustered_indicator_tensor.view(N, H,C, W)
        # print(clustered_indicator_tensor)
        # print(intermediate_tensor)
        clustered_tensor = intermediate_tensor.permute(0, 2, 1, 3)
        
        # Distributing the faults based on the mask
        fault_types = torch.tensor([1, 2, 3, 4], device=self.device)
        probabilities = torch.tensor([self.p_SA00[0], self.p_SA01[0], self.p_SA10[0], self.p_SA11[0]], device=self.device)
        
        for a in range(N):
            for b in range(C):
                true_indices = clustered_tensor[a, b].nonzero(as_tuple=True)
                num_true = true_indices[0].size(0)
                if num_true > 0:
                    faults = torch.multinomial(probabilities, num_true, replacement=True)
                    final_fault_state[a, b][true_indices] = fault_types[faults].to(final_fault_state.dtype)
                
        return final_fault_state


In [23]:
def verify_dist_gen_cluster():
    # Define test parameters
    p_state_shape = (2, 3, 8, 8)  # Example shape
    fault_rate = 0.5 # 25% fault rate
    device = 'cpu'  # Use 'cuda' if testing on GPU
    
    # Initialize the class with the test parameters
    test_class = OptimizedClass(p_state_shape, fault_rate, device=device)
    
    # Generate the faults
    final_fault_state = test_class.dist_gen_cluster()
    # print(final_fault_state.view(p_state_shape[1]*p_state_shape[3],p_state_shape[0]*p_state_shape[2]))
    print(final_fault_state)
    # Verify the fault distribution
    total_elements = final_fault_state.numel()
    expected_faults = int(fault_rate * total_elements)
    actual_faults = torch.count_nonzero(final_fault_state>0).item()
    
    # Verify the fault type distribution based on provided probabilities
    fault_counts = torch.zeros(4, dtype=torch.int32)
    for i in range(1, 5):  # Assuming fault types are {1, 2, 3, 4}
        fault_counts[i-1] = torch.count_nonzero(final_fault_state == i).item()
    
    print(f"Expected number of faults: {expected_faults}")
    print(f"Actual number of faults: {actual_faults}")
    print(f"Fault distribution (for types 1 to 4): {fault_counts.tolist()}")
    
    # Check if the actual number of faults is within a reasonable range of the expected value
    # Note: Due to randomness, the actual number might not match exactly
    assert abs(actual_faults - expected_faults) <= total_elements * 0.05, "Fault count mismatch beyond tolerance."

    # Further checks can be added for specific fault type distributions if needed

# Run the verification
verify_dist_gen_cluster()


[[False False False  True  True  True  True  True]
 [False False False  True  True  True  True  True]
 [False False False  True  True  True  True  True]
 [False  True  True  True  True  True  True  True]
 [False False False  True  True  True  True  True]
 [False False False False  True False  True  True]
 [False False False False False False  True False]
 [False False False False False False  True False]]
tensor([[[[False, False, False,  True,  True,  True,  True,  True],
          [False, False, False,  True,  True,  True,  True,  True],
          [False, False, False,  True,  True,  True,  True,  True],
          [False,  True,  True,  True,  True,  True,  True,  True],
          [False, False, False,  True,  True,  True,  True,  True],
          [False, False, False, False,  True, False,  True,  True],
          [False, False, False, False, False, False,  True, False],
          [False, False, False, False, False, False,  True, False]],

         [[False, False, False,  True,  True,

In [38]:
import torch
import numpy as np

class OptimizedClass:
    def __init__(self, p_state_shape, fault_rate, device='cpu'):
        self.p_state = torch.randn(p_state_shape, device=device)  # Initialize with the given shape on the specified device
        self.fault_rate = fault_rate
        self.device = device
        self.G_shape = p_state_shape  # Assuming G_shape is meant to match p_state_shape
        self.p_SA00 = [0.25]  # Example probability distribution
        self.p_SA01 = [0.25]
        self.p_SA10 = [0.25]
        self.p_SA11 = [0.25]
    # __init__ method remains the same...

    def dist_gen_cluster(self):
        N, C, H, W = self.G_shape
        reshaped_dim = (N*H, C*W)
        
        # Randomly select a variance within the given range
        variance_range = (1, H)
        variance = np.random.uniform(*variance_range)
        
        # Generate a random center within the grid
        random_center = (np.random.randint(0, reshaped_dim[0]), np.random.randint(0, reshaped_dim[1]))
        
        # Generate grid points
        x, y = np.meshgrid(np.arange(reshaped_dim[0]), np.arange(reshaped_dim[1]), indexing='ij')
        
        # Calculate the Gaussian PDF for each point
        pdf = np.exp(-(((x - random_center[0])**2 + (y - random_center[1])**2) / (2 * variance**2)))
        pdf /= pdf.sum()  # Normalize the PDF
        
        # Determine fault states based on the fault rate
        threshold = np.percentile(pdf, 100 - (self.fault_rate * 100))
        fault_state = pdf >= threshold
        print(fault_state.astype(int))
        # Reshape the fault state back to the original tensor shape
        fault_state_reshaped = fault_state.reshape(N, H, C, W).transpose(0, 2, 1, 3)
        # print(fault_state_reshaped.astype(int))
        final_fault_state = torch.tensor(fault_state_reshaped.astype(int), device=self.device)
        
        # Randomly assign fault types based on probabilities
        fault_types = torch.tensor([1, 2, 3, 4], dtype=torch.int, device=self.device)
        probabilities_tensor = torch.tensor([self.p_SA00[0], self.p_SA01[0], self.p_SA10[0], self.p_SA11[0]], dtype=torch.float, device=self.device)
        
        for a in range(N):
            for b in range(C):
                true_indices = final_fault_state[a, b].nonzero(as_tuple=False)
                num_true = true_indices.size(0)
                if num_true > 0:
                    faults = torch.multinomial(probabilities_tensor, num_true, replacement=True)
                    for idx, fault in zip(true_indices, faults):
                        final_fault_state[a, b, idx[0], idx[1]] = fault_types[fault]
        
        return final_fault_state


In [40]:
def verify_dist_gen_cluster():
    # Define test parameters
    p_state_shape = (2, 3, 8, 8)  # Example shape
    fault_rate = 0.5 # 25% fault rate
    device = 'cpu'  # Use 'cuda' if testing on GPU
    
    # Initialize the class with the test parameters
    test_class = OptimizedClass(p_state_shape, fault_rate, device=device)
    
    # Generate the faults
    final_fault_state = test_class.dist_gen_cluster()
    # print(final_fault_state.view(p_state_shape[1]*p_state_shape[3],p_state_shape[0]*p_state_shape[2]))
    print(final_fault_state)
    # Verify the fault distribution
    total_elements = final_fault_state.numel()
    expected_faults = int(fault_rate * total_elements)
    actual_faults = torch.count_nonzero(final_fault_state>0).item()
    
    # Verify the fault type distribution based on provided probabilities
    fault_counts = torch.zeros(4, dtype=torch.int32)
    for i in range(1, 5):  # Assuming fault types are {1, 2, 3, 4}
        fault_counts[i-1] = torch.count_nonzero(final_fault_state == i).item()
    
    print(f"Expected number of faults: {expected_faults}")
    print(f"Actual number of faults: {actual_faults}")
    print(f"Fault distribution (for types 1 to 4): {fault_counts.tolist()}")
    
    # Check if the actual number of faults is within a reasonable range of the expected value
    # Note: Due to randomness, the actual number might not match exactly
    assert abs(actual_faults - expected_faults) <= total_elements * 0.05, "Fault count mismatch beyond tolerance."

    # Further checks can be added for specific fault type distributions if needed

# Run the verification
verify_dist_gen_cluster()


[[0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
tensor([[[[0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 0],
   

In [14]:
import torch

# Example dimensions and tensor
N, C, H, W = 2, 3, 8, 8  # Original dimensions
original_tensor = torch.randint(0, 10, (N, C, H, W), dtype=torch.float32)  # Example tensor

# Example reshaping to (N*H, C*W)
reshaped_tensor = original_tensor.reshape(N*H, C*W)
intermediate_tensor = reshaped_tensor.view(N, H, C, W)
restored_tensor = intermediate_tensor.permute(0, 2, 1, 3)  # Now it should be (N, C, H, W)

print("Restored shape:", restored_tensor)

# To restore the original shape, first initialize an empty list to hold the slices
slices = []

# Iterate over the reshaped tensor to cut it into HxW slices and store them
for n in range(N):
    for c in range(C):
        # Calculate the start and end indices for slicing
        start_row = n * H
        end_row = start_row + H
        start_col = c * W
        end_col = start_col + W
        
        # Slice the tensor and add it to the list
        slice = reshaped_tensor[start_row:end_row, start_col:end_col]
        slices.append(slice.unsqueeze(0))  # Unsqueeze to add a dimension for concatenation

# Concatenate the slices along a new dimension to form a tensor of shape (N*C, 1, H, W)
concatenated_slices = torch.cat(slices, dim=0)

# Finally, reshape to get the original tensor shape (N, C, H, W)
restored_tensor = concatenated_slices.view(N, C, H, W)

# Verify the shapes
print("Original shape:", original_tensor)
print("Reshaped shape:", reshaped_tensor)
print("Restored shape:", restored_tensor)


Restored shape: tensor([[[[4., 7., 0., 5., 0., 6., 2., 1.],
          [7., 2., 9., 7., 3., 3., 6., 7.],
          [5., 8., 9., 0., 8., 6., 4., 7.],
          [0., 3., 7., 5., 0., 5., 2., 1.],
          [7., 4., 3., 0., 4., 6., 5., 5.],
          [1., 1., 8., 9., 0., 0., 1., 0.],
          [2., 3., 3., 7., 3., 2., 6., 7.],
          [9., 9., 3., 5., 8., 2., 1., 9.]],

         [[2., 7., 1., 1., 8., 5., 0., 7.],
          [4., 9., 1., 5., 8., 5., 9., 7.],
          [4., 9., 5., 4., 1., 1., 8., 4.],
          [6., 3., 9., 5., 8., 5., 3., 8.],
          [0., 6., 6., 8., 5., 1., 3., 4.],
          [6., 8., 7., 3., 6., 0., 2., 6.],
          [7., 4., 3., 8., 2., 5., 5., 6.],
          [7., 0., 6., 6., 5., 7., 7., 5.]],

         [[2., 9., 1., 4., 0., 2., 9., 8.],
          [0., 3., 0., 4., 9., 6., 6., 0.],
          [4., 2., 4., 7., 1., 9., 3., 9.],
          [8., 2., 5., 7., 7., 4., 1., 4.],
          [6., 0., 1., 1., 7., 4., 5., 5.],
          [4., 9., 8., 3., 7., 0., 8., 9.],
          [2

In [12]:
e_results = torch.load('cifar-10/resnet18_cifar10_worst_case_error_ec.pth')
ec_results = torch.load('cifar-10/densenet121_cifar10_error_ec.pth')

In [13]:
for i in e_results.keys():
    print(i)
    print(e_results[i])
    #perform average on the list 
    if i == 'fault_0.0':
        avg = sum(e_results[i])/len(e_results[i])
        print(avg)


fault_90.0
[91.08]
fault_50.0
[91.67]
fault_40.0
[91.77]
fault_25.0
[91.79]
fault_10.0
[91.59]
fault_5.0
[91.86]
fault_1.0
[93.86]


In [ ]:
import pandas as pd

# Example dictionary

# Convert the dictionary to a pandas DataFrame

df = pd.DataFrame(e_results)
df_ec = pd.DataFrame(ec_results)
# Specify the Excel file path
file_path = './my_data.xlsx'
file_path2 = './my_data_ec.xlsx'
# Export the DataFrame to an Excel file
df.to_excel(file_path, index=False, engine='openpyxl')
df_ec.to_excel(file_path2, index=False, engine='openpyxl')
print(f'Data successfully written to {file_path}')


In [ ]:
#All the codes are to understand the modelsize, grad size and gradsH size